# Business question

> How can we develop an Automated Valuation Model (AVM) to provide a cold-start asking price for new Airbnb listings based strictly on their spatial features, and which geographic segments present the highest pricing volatility?

# Notebook execution guidelines

A kaggle username and its PAT is needed to retrieve the data, get yours from https://www.kaggle.com/settings/api then generate a "Legacy API Credentials".
This data goes within the `.env` file.

Before running this notebook, initialize a virtual environment and install requirements:
1. `python -m venv .venv`: to create the virtual environment.
2. `.venv\scripts\activate`: to activate the virtual environment using Windows.
3. `pip install -r requirements.txt`: to install the list of requirements.

In [ ]:
import kaggle
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.neighbors import BallTree
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA

RANDOM_SEED = 85

load_dotenv()
datasets_path = Path("dataset")

Download and read file (requires Kaggle account):

In [ ]:
if not (datasets_path / "AB_NYC_2019.csv").exists():
    kaggle.api.dataset_download_files(
        "dgomonov/new-york-city-airbnb-open-data", path=datasets_path, unzip=True
    )
df_airbnb = pd.read_csv(datasets_path / "AB_NYC_2019.csv")

In [ ]:
# City of New York: 2020 Neighborhood Tabulation Areas (NTAs) - Mapped
# https://data.cityofnewyork.us/City-Government/2020-Neighborhood-Tabulation-Areas-NTAs-Mapped/4hft-v355
url = "https://data.cityofnewyork.us/resource/9nt8-h7nd.geojson"
nyc_geojson = gpd.read_file(url)

# convert airbnb dataframe to geodataframe
gdf_airbnb = gpd.GeoDataFrame(
    df_airbnb,
    geometry=gpd.points_from_xy(df_airbnb.longitude, df_airbnb.latitude),
    crs="EPSG:4326",
)
nyc_geojson = nyc_geojson.to_crs("EPSG:4326")  # ensure same coordinate reference system
airbnb_enriched = gpd.sjoin(
    gdf_airbnb,
    nyc_geojson[["ntaname", "boroname", "geometry"]],
    how="left",
    predicate="within",
)  # assign neighborhoods and boroughs to airbnb listings

fig, ax = plt.subplots(figsize=(12, 12))
nyc_geojson.plot(
    ax=ax, color="lightgrey", edgecolor="white", linewidth=0.5
)  # draw base map of NYC neighborhoods
airbnb_enriched.plot(
    ax=ax, color="red", markersize=1, alpha=0.3
)  # Overlay airbnb listings on top of the NYC map
ax.set_title("Airbnb Listings in NYC", fontsize=14)
ax.set_axis_off()

plt.show()

In [ ]:
count_neighborhoods = airbnb_enriched["ntaname"].value_counts().reset_index()
count_neighborhoods.columns = ["ntaname", "total_listings"]

density_map = nyc_geojson.merge(
    count_neighborhoods, on="ntaname", how="left"
)  # join the counts with the GeoDataFrame to prepare for choropleth mapping

density_map["total_listings"] = density_map["total_listings"].fillna(
    0
)  # fill NaN values with 0 for neighborhoods with no listings
fig, ax = plt.subplots(figsize=(12, 12))

density_map.plot(
    column="total_listings",
    cmap="OrRd",
    linewidth=0.3,
    edgecolor="black",
    legend=True,
    legend_kwds={"shrink": 0.6, "label": "Rentals volume by NTA"},
    ax=ax,
)

ax.set_title("Airbnb market concentration by NTA", fontsize=16)
ax.set_axis_off()

plt.show()

# Data Preparation

## Data analysis
Explore how data is composed, then cleanup for ML modeling.

## Data cleaning
1. Remove null and missing values
2. Remove outliers
3. One-Hot Encoding for critical variables
4. Standardise geospatial data

In [ ]:
print("The field name of data: ", df_airbnb.columns)  # The field name of data
print("Number of fields in data: ", len(df_airbnb.columns))  # Number of fields in data
print("Number of data in data: ", len(df_airbnb))  # Number of data in data

print(df_airbnb.info)
display(df_airbnb.head(10))

In [ ]:
df_clean = airbnb_enriched.copy()
df_clean = df_clean.dropna(
    subset=["ntaname"]
)  # drop rows where 'ntaname' is NaN, which indicates listings outside of NYC boundaries
df_clean["reviews_per_month"] = df_clean["reviews_per_month"].fillna(
    0
)  # fill NaN values in 'reviews_per_month' with 0, indicating no reviews for those listings

# removing outliers
max_price = df_clean["price"].quantile(
    0.99
)  # price 0 is an error, removing the top 1% of prices to avoid skewing the model
df_clean = df_clean[(df_clean["price"] > 0) & (df_clean["price"] <= max_price)]
df_clean = df_clean[
    df_clean["minimum_nights"] <= 365
]  # minimum_nights greater than 365 are likely errors or special cases, so we filter them out

# remove unnecesary columns for ML
ml_columns = [
    "ntaname",
    "boroname",
    "latitude",
    "longitude",
    "room_type",
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365",
]

df_ml = df_clean[ml_columns].copy()
df_ml["log_price"] = np.log1p(
    df_ml["price"]
)  # log1p is used to avoid issues with log(0) and to handle the skewness of the price distribution

# Export
df_ml.to_csv(datasets_path / "airbnb_nyc_ml_base.csv", index=False)
df_ml.info()

# Enhance dataset
Adding distance to public transportation, and city tourist attractions. 

In [ ]:
url_mta = "https://data.ny.gov/api/views/i9wp-a4ja/rows.csv?accessType=DOWNLOAD"
df_mta = pd.read_csv(url_mta)
col_lat_mta = (
    "Entrance Latitude" if "Entrance Latitude" in df_mta.columns else "Latitude"
)
col_lon_mta = (
    "Entrance Longitude" if "Entrance Longitude" in df_mta.columns else "Longitude"
)
df_mta = df_mta.dropna(subset=[col_lat_mta, col_lon_mta])
airbnb_coords = np.radians(
    df_ml[["latitude", "longitude"]].values
)  # convert coordiantes to radians
mta_coords = np.radians(
    df_mta[[col_lat_mta, col_lon_mta]].values
)  # convert coordiantes to radians
tree = BallTree(
    mta_coords, metric="haversine"
)  # create space tree based on subway stations
distances_rad, indices = tree.query(
    airbnb_coords, k=1
)  # Find nearest station for each airbnb listing
df_ml["dist_meters_to_subway"] = (
    distances_rad.flatten() * 6371000
)  # Convert radians to meters

# Check
print(df_ml[["latitude", "longitude", "dist_meters_to_subway"]].head())

In [ ]:
# Calculate No-ML Baseline (Median by neighborhood and room type)
df_ml["baseline_log_price"] = df_ml.groupby(["boroname", "room_type"])[
    "log_price"
].transform("median")
baseline_r2 = r2_score(df_ml["log_price"], df_ml["baseline_log_price"])
print(f"R2 Score of No-ML Baseline (Median Lookup): {baseline_r2:.4f}")

In [ ]:
# adding points of interest: we considered "https://data.cityofnewyork.us/City-Government/CommonPlace/rxuy-2muj" but has too many points
# we are going with strategic points of interest that are relevant to tourists and visitors of NYC
pois = [
    ("Times Square", 40.7580, -73.9855),
    ("Central Park", 40.7644, -73.9730),
    ("Met Museum", 40.7794, -73.9632),
    ("Empire State", 40.7484, -73.9857),
    ("Washington Square", 40.7308, -73.9973),
    ("SoHo", 40.7233, -73.9988),
    ("Financial District", 40.7074, -74.0113),
    ("Penn Station", 40.7505, -73.9934),
    ("Grand Central", 40.7527, -73.9772),
    ("Williamsburg", 40.7160, -73.9587),
    ("Barclays Center", 40.6826, -73.9754),
    ("Long Island City", 40.7465, -73.9455),
]
poi_coords = np.radians([[lat, lon] for _, lat, lon in pois])
airbnb_coords = np.radians(df_ml[["latitude", "longitude"]].values)
tree_poi = BallTree(poi_coords, metric="haversine")
dist_rad, _ = tree_poi.query(airbnb_coords, k=1)
df_ml["dist_min_poi_meters"] = dist_rad.flatten() * 6371000

print(df_ml[["latitude", "longitude", "dist_min_poi_meters"]].head())

# Clustering Analysis

The purpose of this clustering analysis is to group Airbnb listings with similar characteristics.  
This helps identify different market segments, such as budget listings, premium listings, highly available listings, and listings close to tourist attractions or public transport.

In [ ]:
# Select numerical features that describe listing price, demand, host activity,
# availability, and location accessibility.
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

clustering_features = [
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365",
    "dist_meters_to_subway",
    "dist_min_poi_meters",
]

df_cluster = df_ml[clustering_features].copy()

# K-Means is distance-based, so all features must be scaled.
# Without scaling, large-value features such as distances or availability would dominate the clustering.
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(df_cluster)

In [ ]:
inertias = []
silhouette_scores = []
k_values = range(2, 9)

# Compare multiple values of k to choose a suitable number of clusters.
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)

    labels = kmeans.fit_predict(X_cluster_scaled)

    # Inertia measures within-cluster compactness and is used for the Elbow Method.
    inertias.append(kmeans.inertia_)

    # Silhouette score measures how well-separated the clusters are.
    # A sample is used because the full dataset makes silhouette calculation slow.
    score = silhouette_score(X_cluster_scaled, labels, random_state=RANDOM_SEED)

    silhouette_scores.append(score)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_values, inertias, marker="o")
plt.xlabel("Number of clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means Clustering")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_values, silhouette_scores, marker="o")
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score by Number of Clusters")
plt.show()

In [ ]:
# Train the final K-Means model using the selected number of clusters.
# k=3 was selected based on the highest Silhouette Score and the Elbow Method.
best_k = 3

kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_SEED, n_init=10)

df_ml["cluster"] = kmeans.fit_predict(X_cluster_scaled)

In [ ]:
# Create a numerical profile for each cluster.
# This helps interpret what each group of listings represents.
cluster_profile = df_ml.groupby("cluster")[clustering_features].mean().round(2)

# Count how many listings belong to each cluster.
cluster_sizes = df_ml["cluster"].value_counts().sort_index()

# Combine average feature values and cluster sizes into one summary table.
cluster_summary = cluster_profile.copy()
cluster_summary["number_of_listings"] = cluster_sizes

cluster_summary

In [ ]:
# Analyze room type distribution within each cluster.
# This helps determine whether clusters represent different property types.
room_type_distribution = pd.crosstab(
    df_ml["cluster"], df_ml["room_type"], normalize="index"
).round(2)

room_type_distribution

In [ ]:
# Analyze borough distribution within each cluster.
# This connects the clustering results to the geographic business question.
borough_distribution = pd.crosstab(
    df_ml["cluster"], df_ml["boroname"], normalize="index"
).round(2)

borough_distribution

In [ ]:
cluster_summary["price"].plot(kind="bar", figsize=(8, 5))

plt.xlabel("Cluster")
plt.ylabel("Average Price")
plt.title("Average Airbnb Price by Cluster")
plt.show()

In [ ]:
cluster_summary["number_of_listings"].plot(kind="bar", figsize=(8, 5))

plt.xlabel("Cluster")
plt.ylabel("Number of Listings")
plt.title("Number of Airbnb Listings by Cluster")
plt.show()

In [ ]:
# Reduce the scaled clustering features to two principal components for visualization.
# PCA is used only for plotting, not for training the K-Means model.
pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_cluster_pca = pca.fit_transform(X_cluster_scaled)

df_ml["pca_1"] = X_cluster_pca[:, 0]
df_ml["pca_2"] = X_cluster_pca[:, 1]

In [ ]:
# Colour-blind friendly palette for the three clusters.
cluster_colors = {
    0: "#0072B2",  # blue
    1: "#000000",  # black
    2: "#009E73",  # green
}

plt.figure(figsize=(8, 6))

for cluster in sorted(df_ml["cluster"].unique()):
    subset = df_ml[df_ml["cluster"] == cluster]
    plt.scatter(
        subset["pca_1"],
        subset["pca_2"],
        color=cluster_colors[cluster],
        label=f"Cluster {cluster}",
        alpha=0.5,
        s=10,
    )

plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title("Airbnb Listing Clusters Visualized with PCA")
plt.legend()
plt.show()

In [ ]:
# Plot clusters geographically to assess whether market segments have spatial patterns.
fig, ax = plt.subplots(figsize=(12, 12))

nyc_geojson.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.5)

for cluster in sorted(df_ml["cluster"].unique()):
    subset = df_ml[df_ml["cluster"] == cluster]
    ax.scatter(
        subset["longitude"],
        subset["latitude"],
        color=cluster_colors[cluster],
        s=5,
        alpha=0.4,
        label=f"Cluster {cluster}",
    )

ax.set_title("Geographic Distribution of Airbnb Listing Clusters", fontsize=14)
ax.set_axis_off()
ax.legend(loc="upper right", markerscale=3)

plt.show()

In [ ]:
# Final encoding for ML - categorical variables to numbers
# we discard ntaname due high cardinality, we keep boroname as a proxy for aggregated location
df_final = pd.get_dummies(
    df_ml.drop(columns=["ntaname"]), columns=["room_type", "boroname"], drop_first=True
)

X = df_final.drop(
    columns=["price", "log_price", "number_of_reviews", "reviews_per_month"]
)
y = df_final["log_price"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=90
)

# train baseline (random forest)
model = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=90)
model.fit(X_train, y_train)

# Initial evaluation
preds = model.predict(X_test)
print(f"R2 Score over base model: {r2_score(y_test, preds):.4f}")

# Analysis of variable importance to justify which variables are effective
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(
    ascending=False
)
print("\nVariable importance:")
print(importances)

# Export
df_final.to_csv(datasets_path / "airbnb_nyc_ml_final.csv", index=False)

In [ ]:
# Out-of-fold estimation
oof_log_price = cross_val_predict(
    model, X, y, cv=KFold(n_splits=5, shuffle=True, random_state=90)
)
df_ml["predicted_price"] = np.expm1(oof_log_price)
df_ml["residue_error"] = df_ml["price"] - df_ml["predicted_price"]

# Maps the mean absolute error per neighborhood to visualize market volatility and unobserved quality impact.
df_ml["absolute_error"] = np.abs(df_ml["residue_error"])
volatility_nta = df_ml.groupby("ntaname")["absolute_error"].mean().reset_index()

volatility_map = nyc_geojson.merge(volatility_nta, on="ntaname", how="left")

fig, ax = plt.subplots(figsize=(12, 12))
volatility_map.plot(
    column="absolute_error",
    cmap="YlOrRd",
    linewidth=0.5,
    edgecolor="black",
    legend=True,
    legend_kwds={"label": "Mean Absolute Error ($) - Market Volatility"},
    missing_kwds={"color": "lightgrey"},
    ax=ax,
)

ax.set_title("Real Estate Volatility by NTA (Unobserved Quality Impact)", fontsize=16)
ax.set_axis_off()
plt.show()

## Real Estate Volatility by NTA Explaination

This choropleth map visualises spatial market volatility by aggregating the mean absolute error of our regression model across neighbourhoods, acting as a risk indicator for the cold-start pricing tool. 
**High-error zones** highlight areas where unobserved property quality heavily dictates value, **demonstrating that baseline spatial features are insufficient for accurate automated valuation**. By intersecting these geographic error margins with the K-Means clustering results, we can mathematically quantify which specific market segments drive this pricing uncertainty, allowing the platform to confidently deploy automated pricing for highly standardised listings whilst flagging volatile segments that necessitate further qualitative data.